# Archived Prototype

This is the original exploratory notebook. It is not used by the application.
See  for the current design and  for the production code.

In [ ]:
!pip install google-generativeai openai

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import datetime as dt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pointbiserialr
from sklearn.ensemble import IsolationForest
import os
import google.generativeai as genai
from google.colab import userdata
from openai import OpenAI
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance
import shap
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('FinancialMarketData.xlsx - EWS.csv')

In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
#Find correlations of features to Y
correlations=df.drop(columns = ['Data'])
print(correlations.corr())
correlations=correlations.drop("Y", axis=1).apply(lambda x: x.corr(df.Y))
correlations = correlations.sort_values(ascending=False)
correlations #Top ascending VIX GTIT2,10,30 EONIA

In [ ]:
client = OpenAI(api_key = userdata.get('OPEN_AI_KEY'))

In [ ]:
prompt = f"""You are an expert financial data analyst. You have a time series dataset of stock prices at certain dates, with 'Data' representing the date, and 'Y' as a binary target that represents whether an anomaly occured in the stock market on that date.
 Your goal is to develop a model that detects anomalies in the stock market based on this time series data. Each feature other than the 'Data' column represents a stock ticker. Let's go through the process step by step.

  Step 1. Your first step is to find what each stock ticker represents in the data
  Step 2. Your second step is to analyze the values of the stock tickers at each date and how the values differ than usual when the binary target('Y') is '1' (1 means anomaly occured on that date).
    This means you should look for outliers in the data such as sudden spikes in the prices of the stock tickers.
    Contextualize anomalous dates from 2000 to 2021 by matching them with dates of known black swan events, such as (but not limited to) the Dot Com Bubble, 2008 Financial Crisis and the COVID-19 pandemic
    In addition to the information from the dataset, you should consider the correlations of each stock ticker to the target column in your analysis. The list of correlations is given here:"{str(list(correlations))}
    These correlations can help you make a more accurate analysis of whether if a stock ticker is more suited for anomaly detection or not.
    Remember not to solely rely on the provided correlations as there may be other factors that can influence a stock ticker's reliability.

  Step 3. Your third step is to rank each of these stock tickers on reliability for anomaly detection. List out the top 15 most reliable and 15 least reliable stock tickers.

  Now, apply these steps in your analysis of the dataset. Do not just make suggestions or an outline, make sure to complete each step. Verify that the information in each step is correct.

  Do not give me hypothetical placeholder values anywhere, only use the tickers in the dataset
  """


user_prompt = f"This is the given dataset: {str(list(df))}"

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "developer", "content": prompt},
        {"role": "user", "content": user_prompt}
    ]
)

# Print the  from the API
message = response.choices[0].message.content

In [ ]:
print(response.choices[0].message.content)

In [ ]:
features = df.drop(columns=['Y'])
features

In [ ]:
target = df['Y']
target

In [ ]:
count = 0
for i in target:
    if i == 1:
        count += 1
print(str((count/len(target))*100),"%")

In [ ]:
#Convert datetime of data['DATA']
features['Data'] = pd.to_datetime(df["Data"])

In [ ]:
features

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(features['Data'], features['VIX'], label='CBOE Volatility Index (VIX)', color='green', linewidth=2)
plt.title('Most Correlated Trait(VIX)', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Price (USD)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='Y', y='VIX', data=df)

plt.title('Violin Plot of VIX by Target (Y)', fontsize=14)
plt.xlabel('Target (Y)', fontsize=12)
plt.ylabel('VIX', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='Y', y='DXY', data=df)

plt.title('Violin Plot of DXY by Target (Y)', fontsize=14)
plt.xlabel('Target (Y)', fontsize=12)
plt.ylabel('DXY', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(features['Data'], features['DXY'], label='DXY', color='green', linewidth=2)
plt.title('USD Index', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Price (USD)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(features['Data'], features['XAU BGNL'], label='Cost of Gold (XAU)', color='green', linewidth=2)
plt.title('Gold Index', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Price (USD)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(30, 8))
plot_features = ['VIX', 'CRY', 'Cl1', 'ECSURPUS', 'GTITL2YR' , 'MXJP']
fig, axes = plt.subplots(1, len(plot_features), figsize=(30, 6))
fig.suptitle('Feature Distributions By Target Class (Y)')


for i, column in enumerate(plot_features):
    sns.violinplot(x='Y', y=column, data=df, ax=axes[i], inner = 'box')

    axes[i].set_title(f'Violin Plot of {column} by Target (Y)')
    axes[i].set_xlabel('Target (Y)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
#plt.plot(train_data['Data'], train_data['MXJP'], label='MXJP', color='red', linewidth=2)
plt.plot(features['Data'], features['MXJP'], label='MXJP', color='green', linewidth=2)
plt.title('Least Correlated trait (MXJP)', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Price (USD)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

Simple LR

In [ ]:
train_data = features.drop(columns=['Data'])

In [ ]:
#Drop least correlated features
"""Based on hypothetical results:
Based on the provided correlations, stock tickers are ranked for their reliability in detecting anomalies. Here are the top 15 and least reliable 15:

**Top 15 Most Reliable Tickers (highest to lowest correlation with 'Y'):**
1. `XAU BGNL`: 0.5984
2. `ECSURPUS`: 0.3045
3. `BDIY`: 0.2976
4. `CRY`: 0.2950
5. `DXY`: 0.1803
6. `JPY`: 0.1415
7. `GBP`: 0.1399
8. `Cl1`: 0.1337
9. `VIX`: 0.1306
10. `USGG30YR`: 0.1146
11. `GT10`: 0.1123
12. `USGG2YR`: 0.1032
13. `USGG3M`: 0.1001
14. `US0001M`: 0.0998
15. `GTDEM30Y`: 0.0350

**15 Least Reliable Tickers (lowest to highest correlation with 'Y'):**
1. `MXCN`: -0.2794
2. `MXIN`: -0.2539
3. `MXRU`: -0.2098
4. `MXBR`: -0.1972
5. `GTJPY2YR`: -0.1939
6. `GTJPY10YR`: -0.1897
7. `GTJPY30YR`: -0.1889
8. `MXJP`: -0.1695
9. `MXEU`: -0.1566
10. `MXUS`: -0.1535
11. `GTGBP2Y`: -0.1512
12. `GTGBP20Y`: -0.1496
13. `GTGBP30Y`: -0.1488
14. `GTITL2YR`: -0.1436
15. `GTITL10YR`: -0.1049

The correlations indicate which stock tickers are more strongly associated with anomalies. However, further analysis and context consideration are needed for a comprehensive model for anomaly detection.
"""


In [ ]:
train_data = train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10','CRY','Cl1','GTITL2YR', 'XAU BGNL']]

In [ ]:
train_data

In [ ]:
train_data, target

# Finding top Features

Test to find most important features from Logistic Regression, XGBoost, Isolation Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_data, target, test_size=0.2, random_state=42)

In [ ]:
SMOTE = SMOTE(random_state=42)
X_train, y_train = SMOTE.fit_resample(X_train, y_train)#Smote makes it worse

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [ ]:
lr_model = LogisticRegression(solver = 'liblinear', C = 2.0, max_iter = 1000, class_weight = 'balanced')

In [ ]:
def evaluate_and_save_model(model, X_train, X_test, y_train, y_test, filename):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{model.__class__.__name__} Accuracy: {accuracy:.4f}")

    print(f"\nClassification Report:\n{classification_report(y_test,y_pred)}")
    print("------")

    with open(filename, "wb") as file:
      pickle.dump(model, file)

      print(f"Model saved as: {filename}")

In [ ]:
evaluate_and_save_model(lr_model, X_train, X_test, y_train, y_test, "lr_model.pkl")

In [ ]:
model_fi = permutation_importance(lr_model, X_train, y_train)
model_fi['importances_mean']

In [ ]:
feature_names = train_data.columns
feature_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model_fi['importances_mean']
})
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)
print(feature_importances)

In [ ]:
train_data.columns #If Thhe above outputs very good features, get them

Isolation Forest

In [ ]:
iso_model = IsolationForest(n_estimators=50, max_samples='auto', contamination=0.21, max_features=1.0)
iso_model.fit(train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']])


In [ ]:
#Default training data for testing
train_data =train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']]

In [ ]:
train_data['scores']=iso_model.decision_function(train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']])
train_data['anomaly']=iso_model.predict(train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']])

In [ ]:
train_data['anomaly'] = train_data['anomaly'].replace({-1: 1, 1: 0})

In [ ]:
print("Percentage of outliers in data: {:.2f}%".format(len(train_data[train_data['anomaly'] == 1]) / len(train_data) * 100))

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
print(classification_report(target, train_data['anomaly']))

In [ ]:
exp = shap.TreeExplainer(iso_model) #Explainer
shap_values = exp.shap_values(train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']])

In [ ]:
shap.summary_plot(shap_values,train_data[['BDIY', 'DXY', 'JPY', 'VIX', 'GT10', 'CRY', 'Cl1', 'GTITL2YR',
       'XAU BGNL']], max_display=train_data.shape[1])

# **Feature Engineering and Applying Methods**


In [ ]:
df = pd.read_csv('FinancialMarketData.xlsx - EWS.csv', parse_dates=['Data'])

##Moving Average and Splitting Method

In [ ]:
df_rolling = df[['Data', 'DXY', 'JPY', 'VIX', 'GTITL2YR',
       'XAU BGNL', 'Y']]

In [ ]:
def apply_rolling_average(df, window):
    excluded_columns = {'Month', 'Year', 'Day', 'Data', 'Y', 'USD_Over_100'}
    for col in df.columns:
        if col not in excluded_columns:
            df[f"{col}_rolling_avg_{window}"] = df[col].rolling(window=window).mean()

    return df


In [ ]:
apply_rolling_average(df_rolling, 5)
apply_rolling_average(df_rolling, 20)

In [ ]:
def apply_cross(df, w1, w2):
  excluded_columns = {'Month', 'Year', 'Day', 'Data', 'Y', 'USD_Over_100'}
  for col in df.columns:
    if col not in excluded_columns and "rolling_avg" not in col:
      df[f"{col}_cross"] = (df[f"{col}_rolling_avg_{w1}"] > df[f"{col}_rolling_avg_{w2}"]).astype(int)
  return df

In [ ]:
apply_cross(df_rolling, 5, 20)

In [ ]:
df_rolling['USD_Over_100'] = df_rolling['DXY'] > 100

In [ ]:
df_rolling = df_rolling[['Data', 'DXY', "JPY", "VIX", "GTITL2YR", "XAU BGNL", "Y", "XAU BGNL_cross", "VIX_cross", "USD_Over_100"]]

In [ ]:
corr = df_rolling.corr()['Y']
print(corr)


In [ ]:
def split_timeseries(df, train_years, val_years, total_years ):
  df['Data'] = pd.to_datetime(df['Data'])
  df = df.sort_values('Data')

  train_size = train_years*52
  test_start = train_size + val_years*52
  total_weeks = total_years*52
  if total_years > len(df):
    raise ValueError("Total years cannot be greater than the length of the DataFrame.")

  training_data = df.iloc[:train_size]
  validation_data = df.iloc[train_size:test_start ]
  test_data = df.iloc[test_start: total_weeks]

  print(f"Training data starts from {training_data['Data'].min()} and ends on {training_data['Data'].max()}")
  print(f"Validation data starts from {validation_data['Data'].min()} and ends on {validation_data['Data'].max()}")
  print(f"Test data starts from {test_data['Data'].min()} and ends on {test_data['Data'].max()}")

  return training_data, validation_data, test_data

In [ ]:
train_split, val_split, test_split = split_timeseries(df_rolling, train_years = 9, val_years = 2, total_years = 12)

In [ ]:
train_split = train_split.drop(columns=['Data'])
val_split = val_split.drop(columns=['Data'])
test_split = test_split.drop(columns=['Data'])

In [ ]:
X_train = train_split.drop(columns=['Y'])
y_train = train_split['Y']

X_val = val_split.drop(columns=['Y'])
y_val = val_split['Y']

X_test = test_split.drop(columns=['Y'])
y_test = test_split['Y']


In [ ]:
X_val

In [ ]:
X_train.columns

In [ ]:
lr_model = LogisticRegression(C=0.1, class_weight='balanced')
lr_model.fit(X_train, y_train)

In [ ]:
val_predictions = lr_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_predictions))
print(f"\nClassification Report:\n{classification_report(y_val,val_predictions)}")
print("------")


# 5. Test the Model
test_predictions = lr_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print(f"\nClassification Report:\n{classification_report(y_test,test_predictions)}")
print("------")

Training


In [ ]:
import json

In [ ]:
def save_evaluation(model, X_train, X_test, y_train, y_test, json_file_path):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report_dict = classification_report(y_test,y_pred)
    with open(json_file_path, "w") as json_file:
      json.dump(report_dict, json_file, indent=4)
    print(f"Metrics saved as: {json_file_path}")


In [ ]:
def evaluate_and_save_model(model, X_train, X_test, y_train, y_test, filename):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{model.__class__.__name__} Accuracy: {accuracy:.4f}")

    print(f"\nClassification Report:\n{classification_report(y_test,y_pred)}")
    print("------")

    with open(filename, "wb") as file:
      pickle.dump(model, file)

      print(f"Model saved as: {filename}")

In [ ]:
save_evaluation(lr_model, X_train, X_test, y_train, y_test, "lr_model_features.json")

In [ ]:
evaluate_and_save_model(lr_model, X_train, X_test, y_train, y_test, "lr_model_features.pkl")

In [ ]:
iso_model = IsolationForest(n_estimators=500, contamination='auto', max_features=1.0, random_state = 42, warm_start = True, n_jobs = 1)
iso_features = ["VIX","USD_Over_100", "XAU BGNL_cross"]

In [ ]:
iso_model.fit(X_train[iso_features], y_train)

In [ ]:
val_predictions = iso_model.predict(X_val[iso_features])
val_predictions = np.where(val_predictions == -1, 1, 0)
print("Validation Accuracy:", accuracy_score(y_val, val_predictions))
print(f"\nClassification Report:\n{classification_report(y_val,val_predictions)}")
print("------")


# 5. Test the Model
test_predictions = iso_model.predict(X_test[iso_features])
test_predictions = np.where(test_predictions == -1, 1, 0)
print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print(f"\nClassification Report:\n{classification_report(y_test,test_predictions)}")
print("------")

In [ ]:
exp = shap.TreeExplainer(iso_model) #Explainer
shap_values = exp.shap_values(X_train[iso_features])

In [ ]:
shap.summary_plot(shap_values,X_train[iso_features], max_display=X_train.shape[1])

In [ ]:
X_train

In [ ]:
with open("iso_model.pkl", "wb") as file:
      pickle.dump("iso_model.pkl", file)



##Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=100, random_state=42)

In [ ]:
X_train.columns

In [ ]:
dt_features = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross',
       'VIX_cross', 'USD_Over_100']

In [ ]:
dt_model.fit(X_train[dt_features], y_train)

In [ ]:
val_predictions = dt_model.predict(X_val[dt_features])
print("Validation Accuracy:", accuracy_score(y_val, val_predictions))
print(f"\nClassification Report (Validation):\n{classification_report(y_val, val_predictions)}")
print("------")

test_predictions = dt_model.predict(X_test[dt_features])
print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print(f"\nClassification Report (Test):\n{classification_report(y_test, test_predictions)}")
print("------")

In [ ]:
with open("dt_model.pkl", "wb") as file:
      pickle.dump("dt_model.pkl", file)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_features = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross',
       'VIX_cross', 'USD_Over_100']

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=500,    # Number of trees in the forest
    max_depth=100,      # Maximum depth of each tree
    random_state=42,     # Ensures reproducibility
    n_jobs=-1            # Use all available cores for training
)

rf_model.fit(X_train[rf_features], y_train)

val_predictions = rf_model.predict(X_val[rf_features])
print("Validation Accuracy:", accuracy_score(y_val, val_predictions))
print(f"\nClassification Report (Validation):\n{classification_report(y_val, val_predictions)}")
print("------")

test_predictions = rf_model.predict(X_test[rf_features])
print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print(f"\nClassification Report (Test):\n{classification_report(y_test, test_predictions)}")
print("------")


In [ ]:
with open("rf_model.pkl", "wb") as file:
      pickle.dump("rf_model.pkl", file)

##Milestone 2

In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf
import datetime
import joblib
import google.generativeai as genai
google_api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=google_api_key)

In [ ]:
top_features = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross',
       'VIX_cross', 'USD_Over_100']

In [ ]:
vix_ticker = yf.Ticker("^VIX")
today = datetime.date.today().strftime("%Y-%m-%d")
vix_data = vix_ticker.history(start="2021-01-05", end=today)
vix_data.tail()

In [ ]:
vix_data= vix_data.drop(columns=['Open', 'High', 'Low', 'Volume', 'Dividends', 'Stock Splits'])

In [ ]:
vix_data

In [ ]:
tickers = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL']
ranges = {
    'DXY': (90, 110),
    'JPY': (100, 150),
    'VIX': (10, 40),
    'GTITL2YR': (0.5, 5.0),
    'XAU BGNL': (1700, 2000)
}
start_date = dt.datetime(2021, 1, 5)
end_date = dt.datetime(2024, 12, 31)

In [ ]:
dates = pd.date_range(start=start_date, end=end_date, freq="W-TUE")
data = {"Data": dates}

In [ ]:
data

In [ ]:
for ticker in tickers:
    data[ticker] = np.random.uniform(*ranges[ticker], len(dates))

In [ ]:
new_df = pd.DataFrame(data)
new_df

In [ ]:
new_df['USD_Over_100'] = new_df['DXY'] > 100
new_df['VIX_rolling_avg_5'] = new_df['VIX'].rolling(window=5).mean()
new_df['VIX_rolling_avg_20'] = new_df['VIX'].rolling(window=20).mean()
new_df["VIX_cross"] = (new_df["VIX_rolling_avg_5"] > new_df["VIX_rolling_avg_20"]).astype(int)
new_df['XAU BGNL_rolling_avg_5'] = new_df['XAU BGNL'].rolling(window=5).mean()
new_df['XAU BGNL_rolling_avg_20'] = new_df['XAU BGNL'].rolling(window=20).mean()
new_df["XAU BGNL_cross"] = (new_df["XAU BGNL_rolling_avg_5"] > new_df["XAU BGNL_rolling_avg_20"]).astype(int)

In [ ]:
new_df

In [ ]:
test_lr_model = joblib.load("lr_model_features.pkl")
lr_features = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross',
       'VIX_cross', 'USD_Over_100']
predictions = test_lr_model.predict(new_df[lr_features])

In [ ]:
predictions

In [ ]:
new_df['Target'] = predictions
new_df

In [ ]:
def generate_investment_strategy(report, ticker, predictions_df, model_name):
    prompt = f"""
    You are an expert financial analyst tasked with proposing the most optimal investment strategy for a stock ticker based on the predictions of the given model.

    The model performs a binary classification task on stock ticker features and identifies anomalies that occurred on certain dates. A value of **1** in the target column indicates an anomaly occurred, while **0** indicates no anomaly.
    This is the dataframe containing the features that the model was trained on and the predictions that the model made: {predictions_df}, refer to this when looking for information on the predictions of the model.

    Your response must be structured into three sections:
    1. Explain the model's effectiveness using its metrics. Focus on the model's ability to predict anomalous behavior given it's accuracy, and precision + and recall for the target class based on the report given in the user prompt.
       Be sure to list the stock tickers in the predictions mentioned above, which are the same as the stock tickers the model was trained on. Always refer to the report given in the user prompts, it contains all the metrics. Do not mention the size of the data,
       just state how effective the model will be when providing a strategy.
    2. Analyze trends in the predictions and provide actionable investment strategies (e.g., buy/sell recommendations) of the given stock ticker:{ticker} along with the information given in the predictions.
      Your format should follow:
        **Selected Ticker: {ticker}**
          **Analysis of anomalous days and the behavior of that ticker on anomalous days**
    3. Suggest online resources for staying updated on this stock ticker.

    """


    model = genai.GenerativeModel(model_name=model_name)
    response = model.generate_content([prompt, report])
    return response.text


In [ ]:
def open_json_report(file_path):
    with open(file_path, 'r') as file:
        report = json.load(file)
    return report

In [ ]:
test_lr_report = open_json_report("lr_model_features.json")

In [ ]:
strategy = generate_investment_strategy(test_lr_report, "VIX", new_df, "gemini-1.5-flash")
print(strategy)

In [ ]:
!pip install streamlit pyngrok python-dotenv langchain_openai

In [ ]:
from threading import Thread
from pyngrok import ngrok

In [ ]:
ngrok_token = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(ngrok_token)

In [ ]:
def run_streamlit():
  os.system("streamlit run /content/app.py --server.port 8501")

In [ ]:
%%writefile app.py
import streamlit as st
from langchain_openai.chat_models import ChatOpenAI
from google.colab import userdata
from openai import OpenAIError
import os
from io import StringIO
import json
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAIError
from charset_normalizer import detect
import pandas as pd
import numpy as np
import datetime as dt
import joblib
import google.generativeai as genai

load_dotenv()
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

df = None
st.title("Investment Strategy Generator")
openai_api_key = st.sidebar.text_input("GROQ API Key", type="password")
if openai_api_key:
  os.environ["GROQ_API_KEY"] = openai_api_key
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

def generate_dataframe(tickers, model_file):
  ranges = {
    'DXY': (90, 110),
    'JPY': (100, 150),
    'VIX': (10, 40),
    'GTITL2YR': (0.5, 5.0),
    'XAU BGNL': (1700, 2000)
  }
  start_date = dt.datetime(2021, 1, 5)
  end_date = dt.datetime(2024, 12, 31)
  dates = pd.date_range(start=start_date, end=end_date, freq="W-TUE")
  data = {"Data": dates}
  for ticker in tickers:
    data[ticker] = np.random.uniform(*ranges[ticker], len(dates))
  new_df = pd.DataFrame(data)

  new_df['USD_Over_100'] = new_df['DXY'] > 100
  new_df['VIX_rolling_avg_5'] = new_df['VIX'].rolling(window=5).mean()
  new_df['VIX_rolling_avg_20'] = new_df['VIX'].rolling(window=20).mean()
  new_df["VIX_cross"] = (new_df["VIX_rolling_avg_5"] > new_df["VIX_rolling_avg_20"]).astype(int)
  new_df['XAU BGNL_rolling_avg_5'] = new_df['XAU BGNL'].rolling(window=5).mean()
  new_df['XAU BGNL_rolling_avg_20'] = new_df['XAU BGNL'].rolling(window=20).mean()
  new_df["XAU BGNL_cross"] = (new_df["XAU BGNL_rolling_avg_5"] > new_df["XAU BGNL_rolling_avg_20"]).astype(int)

  test_model = joblib.load(model_file)
  st.write(model_file)
  features = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross','VIX_cross', 'USD_Over_100']
  predictions = test_model.predict(new_df[features])
  new_df['Target'] = predictions
  if model_file == "iso_model.pkl":
    return new_df[['VIX', 'VIX_cross', 'XAU BGNL_cross', 'Target']]
  return new_df[['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL', 'XAU BGNL_cross','VIX_cross', 'USD_Over_100', 'Target']]

def generate_investment_strategy(report, ticker, predictions_df, model_name):
    prompt = f"""
    You are an expert financial analyst tasked with proposing the most optimal investment strategy for a stock ticker based on the predictions of the given model.

    The model performs a binary classification task on stock ticker features and identifies anomalies that occurred on certain dates. A value of **1** in the target column indicates an anomaly occurred, while **0** indicates no anomaly.
    This is the dataframe containing the features that the model was trained on and the predictions that the model made: {predictions_df}, refer to this when looking for information on the predictions of the model.

    Your response must be structured into three sections:
    1. Explain the model's effectiveness using its metrics. Focus on the model's ability to predict anomalous behavior given it's accuracy, and precision + and recall for the target class based on the report given in the user prompt.
       Be sure to list the stock tickers in the predictions mentioned above, which are the same as the stock tickers the model was trained on. Always refer to the report given in the user prompts, it contains all the metrics. Do not mention the size of the data,
       just state how effective the model will be when providing a strategy.
    2. Analyze trends in the predictions and provide actionable investment strategies (e.g., buy/sell recommendations) of the given stock ticker:{ticker} along with the information given in the predictions.
      Your format should follow:
        **Selected Ticker: {ticker}**
          **Analysis of anomalous days and the behavior of that ticker on anomalous days**
    3. Suggest online resources for staying updated on this stock ticker.

    """


    model = genai.GenerativeModel(model_name=model_name)
    response = model.generate_content([prompt, report])
    return response.text

test_report = None
ticker = ""
model_file = ""
json_file = ""
available_tickers = []
if openai_api_key:
  options = ["Random Forest", "Isolation Forest", "Decision Tree", "Logistic Regression"]
  selected_option = st.selectbox("Choose a Model", options)
  if selected_option == "Random Forest":
    model_file = "rf_model.pkl"
    available_tickers = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL']
    json_file = "rf_model.json"
  elif selected_option == "Isolation Forest":
    model_file = "iso_model.pkl"
    available_tickers = ["VIX"]
    json_file = "iso_model.json"
  elif selected_option == "Decision Tree":
    model_file = "dt_model.pkl"
    available_tickers = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL']
    json_file = "dt_model.json"
  elif selected_option == "Logistic Regression":
    model_file = "lr_model_features.pkl"
    available_tickers = ['DXY', 'JPY', 'VIX', 'GTITL2YR', 'XAU BGNL']
    json_file = "lr_model_features.json"
  if selected_option:
    st.write(f"Available for this strategy:{available_tickers}")
    ticker = st.text_input("Enter a stock ticker")
    df = generate_dataframe(available_tickers, model_file)

def chatbot_function(strategy, user_input):
  prompt = f"""You are an expert financial analyst, who is tasked with answering your client's questions on this investment strategy:{strategy}"""
  model = genai.GenerativeModel(model_name="gemini-1.5-flash")
  response = model.generate_content([prompt, user_input])
  return response.text


def open_json_report(file_path):
    with open(file_path, 'r') as file:
        report = json.load(file)
    return report
investment_strategy = None
if json_file and ticker:
  investment_strategy = generate_investment_strategy(open_json_report(json_file), ticker, df, "gemini-1.5-flash")

  st.write(investment_strategy)
  st.session_state.messages.append({
        "role": "assistant",
        "content": investment_strategy
    })

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if text := st.chat_input(""):
    st.session_state.messages.append({"role": "user", "content": text})
    with st.chat_message("user"):
        st.markdown(text)

    with st.chat_message("assistant"):
        response = st.write_stream(iter([chatbot_function(investment_strategy, text)]))
    st.session_state.messages.append({"role": "assistant", "content": response})


In [ ]:
thread = Thread(target=run_streamlit)
thread.start()

In [ ]:
public_url = ngrok.connect(addr='8501', proto='http', bind_tls=True)
print("Public URL:", public_url)

In [ ]:
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
  print(f"Closing tunnel: {tunnel.public_url}->{tunnel.config['addr']}")
  ngrok.disconnect(tunnel.public_url)

In [ ]:
%%writefile .env
GOOGLE_API_KEY=
GROQ_API_KEY